# 6. Mini proje: ürün yönetimi

Bu örnek; veri sınıfı, CSV deposu, envanter servisi ve para hesabını bir araya getirir.
Notebook'u kendi klasöründen çalıştırın.

In [ ]:
import csv
from dataclasses import dataclass
from decimal import Decimal
from pathlib import Path


@dataclass(frozen=True, slots=True)
class Product:
    name: str
    price: Decimal
    quantity: int

    def __post_init__(self) -> None:
        if not self.name.strip():
            raise ValueError("Ürün adı boş olamaz.")
        if self.price < 0:
            raise ValueError("Fiyat negatif olamaz.")
        if self.quantity < 0:
            raise ValueError("Adet negatif olamaz.")

    def total_price(self) -> Decimal:
        return self.price * self.quantity


class ProductRepository:
    @staticmethod
    def from_csv(file_path: Path) -> list[Product]:
        with file_path.open(encoding="utf-8", newline="") as file:
            rows = csv.DictReader(file)
            return [
                Product(
                    name=row["name"],
                    price=Decimal(row["price"]),
                    quantity=int(row["quantity"]),
                )
                for row in rows
            ]


class Inventory:
    def __init__(self, products: list[Product]) -> None:
        self.products = list(products)

    def subtotal(self) -> Decimal:
        return sum((product.total_price() for product in self.products), Decimal("0"))

    def total_with_tax(self, tax_rate: Decimal) -> Decimal:
        return self.subtotal() * (Decimal("1") + tax_rate)

In [ ]:
data_file = Path("data/products.csv")
products = ProductRepository.from_csv(data_file)
inventory = Inventory(products)

for product in products:
    print(f"{product.name}: {product.total_price():,.2f} TL")

print(f"Ara toplam: {inventory.subtotal():,.2f} TL")
print(f"KDV dahil: {inventory.total_with_tax(Decimal('0.20')):,.2f} TL")